# 01 — Data inspection

This notebook inventories the original Al–Cu laser-welding files before any cleaning. It does **not** modify or overwrite source data.

Place the four original files in `data/raw/` before running the inspection cells.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
print("Project root:", PROJECT_ROOT)
print("Raw data folder exists:", RAW_DIR.exists())
print("Files found:")
for path in sorted(RAW_DIR.glob("*")):
    if path.is_file():
        print(f"- {path.name} ({path.stat().st_size:,} bytes)")

## Expected source files

- `Laser Welding.xlsx`
- `Al-Cu_Wobbling_Welding.xlsx`
- `first.txt` (Shin et al. extraction)
- `second..txt` (Lee et al. extraction)

If a file is absent, upload it to `data/raw/` and rerun the inventory.

In [ ]:
expected = ["Laser Welding.xlsx", "Al-Cu_Wobbling_Welding.xlsx", "first.txt", "second..txt"]
present = {p.name for p in RAW_DIR.glob("*") if p.is_file()} if RAW_DIR.exists() else set()
for name in expected:
    print(("FOUND  " if name in present else "MISSING") + name)

## Inspect Excel workbooks

The next cell prints sheet names, dimensions, column names, and a small preview. It reads the files only.

In [ ]:
excel_files = ["Laser Welding.xlsx", "Al-Cu_Wobbling_Welding.xlsx"]
for filename in excel_files:
    path = RAW_DIR / filename
    if not path.exists():
        print(f"\nSKIPPING {filename}: file not found")
        continue
    print(f"\n{'='*80}\n{filename}")
    book = pd.ExcelFile(path)
    print("Sheets:", book.sheet_names)
    for sheet in book.sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        print(f"\nSheet: {sheet} | rows={len(df)} | columns={len(df.columns)}")
        print("Columns:", list(df.columns))
        display(df.head(8))

## Read extracted text files

Review the extracted tables and compare questionable values against the original publications before accepting them into a model dataset.

In [ ]:
for filename in ["first.txt", "second..txt"]:
    path = RAW_DIR / filename
    if path.exists():
        print(f"\n{'='*80}\n{filename}\n")
        print(path.read_text(encoding="utf-8", errors="replace")[:12000])
    else:
        print(f"\nSKIPPING {filename}: file not found")

## Initial data-quality checks

For each Excel sheet, inspect missing cells and exact duplicate rows. These are diagnostic counts only; do not delete anything based on them alone.

In [ ]:
for filename in excel_files:
    path = RAW_DIR / filename
    if not path.exists():
        continue
    print(f"\n{filename}")
    for sheet in pd.ExcelFile(path).sheet_names:
        df = pd.read_excel(path, sheet_name=sheet)
        print(f"\n[{sheet}] rows={len(df)}; exact duplicate rows={df.duplicated().sum()}")
        missing = df.isna().sum()
        print("Missing by column:\n", missing[missing.gt(0)].to_string() or "None")

## Before moving to cleaning

1. Confirm all four source files are present.
2. Check that the sheet previews match your original workbooks.
3. Record any strange values, missing measurements, or unclear units.
4. Do not replace missing values with zero or remove failed-weld records automatically.

The next notebook will create cleaned copies while preserving raw source files and documenting every transformation.